# TP 1 ► Module : Apprentissage Profond (Deep Learning)
> Pr. Gaceb<br>
> S2-Master I2A Pro

**L’Objectif de cette première série de TP** est de Configurer un environnement de développement avec PyCharm, ansi que l'installation des bibliothèques qui serviront au deep learning toute en se familiarisant avec eux*

Les deux bibliothèques les plus utilisées dans ce domaine sont incontestablement **PyTorch (Meta/Facebook)** et **TensorFlow (Google)**.

PyTorch est actuellement la bibliothèque favorite de la recherche et du monde académique, permettant de comprendre en profondeur la structure de vos modèles avec une syntaxe intuitive. Son code ressemble à du Python classique et utilise des graphes de calcul dynamiques, ce qui rend le débogage beaucoup plus simple.

Cette bibliothèque qui sera utilisée majoritairement en TP est idéale pour le prototypage rapide, les projets innovants et l'apprentissage. En revanche, TensorFlow (via Keras) est la référence dominante dans l'industrie et le déploiement en production, permettant de construire des modèles fonctionnels très rapidement avec un minimum de lignes de code.

# Exo 1 - Environment Setup

In [2]:
import torch

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.15.0+cu132
Device: cuda


# Exo 2 $\rightarrow$ 5 [Tensors Tutorial](../../tutorials/Tensors.ipynb)

# Exo 6 - Mise en œuvre de la Descente de Gradient avec PyTorch : Optimisation d'une fonction convexe de degré 2

Pour comprendre la "magie" derrière l'apprentissage automatique et profond. On va utiliser PyTorch pour trouver le minimum de la fonction $f(x)=x^2-4x+4$ avec l’algorithme de la décente de gradient ($x_0=10$, taux d’apprentissage `lr = 0.1`). Mathématiquement, cette fonction est $(x-2)^2$ , donc le minimum est à $2$.

1. **Initialisation :** Déclarez une variable PyTorch `x` initialisée à 10 Assurez-vous que PyTorch puisse calculer les gradients par rapport à cette variable.

In [68]:
x = torch.tensor(10.0, requires_grad=True)
test = x**2  # derivative is 2x
print(f"{x=}")
print(f"{test=}")

x=tensor(10., requires_grad=True)
test=tensor(100., grad_fn=<PowBackward0>)


In [69]:
test.backward()
print(f"{x.grad=}")  # expecting 2 * 10 = 20 for x=10

x.grad=tensor(20.)


2. Appliquer la descente de gradient classique à l’aide d’une boucle à 50 itérations. À chaque étape :
    - Calculez la valeur de la fonction $f(x)=x^2-4x+4$.
    - Déclenchez le calcul des gradients (backward).
    - Mettez à jour x en utilisant la formule mathématique de la descente de gradient :
    $$ x=x – lr × gradient $$
    Afficher l’évolution de `x` chaque 10 itérations.

**Note :** Utilisez with `torch.no_grad()`; pour la mise à jour, et n'oubliez pas de remettre le gradient à zéro.

In [80]:
x = torch.tensor(10.0, requires_grad=True)
lr = 0.1

# 2. Gradient descent loop
for i in range(50):
    f = x**2 - 4*x + 4          # forward pass: compute f(x)
    f.backward()                # backward pass: computes df/dx and stores it in x.grad

    with torch.no_grad():       # the update must not be tracked by autograd (x has `requires_grad=True`)
        x -= lr * x.grad

    x.grad.zero_()              # reset the gradient, otherwise it accumulates

    if (i + 1) % 10 == 0:
        print(f"Iteration {i+1} | x = {x.item():.6f} | f(x) = {f.item():.6f}")

Iteration 10 | x = 2.858993 | f(x) = 1.152921
Iteration 20 | x = 2.092234 | f(x) = 0.013292
Iteration 30 | x = 2.009903 | f(x) = 0.000153
Iteration 40 | x = 2.001063 | f(x) = 0.000002
Iteration 50 | x = 2.000114 | f(x) = 0.000000


3. L'automatisation avec Optimizer : Réinitialisez `x` à 10, utilisez cette fois l'objet [torch.optim.SGD](https://docs.pytorch.org/docs/2.15/generated/torch.optim.SGD.html) pour automatiser la mise à jour à l’aide de la méthode de descente de
gradient stochastique.
- Remplacez votre formule de DG classique par `optimizer.step()`.
- Remplacez la remise à zéro manuelle par `optimizer.zero_grad()`.

Let's define a function for experimenting with different optimizers, learning rates, and number of iterations.

In [78]:
def with_optim(Optimizer, lr=0.1, iterations=50):
    print(f"\n{Optimizer=}\n{iterations=}\n{lr=}", end="\n\n")
    x = torch.tensor(10.0, requires_grad=True)
    optimizer = Optimizer([x], lr=lr)
    
    for i in range(iterations):
        optimizer.zero_grad()
        f = x**2 - 4*x + 4          # forward pass: compute f(x)
        f.backward()                # backward pass
    
        optimizer.step()
    
        if (i + 1) % 10 == 0:
            print(f"Iteration {i+1} | x = {x.item():.6f} | f(x) = {f.item():.6f}")

With the `SGD` optimizer:

In [79]:
with_optim(torch.optim.SGD)


Optimizer=<class 'torch.optim.sgd.SGD'>
iterations=50
lr=0.1

Iteration 10 | x = 2.858993 | f(x) = 1.152921
Iteration 20 | x = 2.092234 | f(x) = 0.013292
Iteration 30 | x = 2.009903 | f(x) = 0.000153
Iteration 40 | x = 2.001063 | f(x) = 0.000002
Iteration 50 | x = 2.000114 | f(x) = 0.000000


4. Application de l’optimiseur Adam : Réinitialisez une nouvelle fois `x` à 10.0.
- Au lieu de SGD, utilisez l'optimiseur [torch.optim.Adam](https://docs.pytorch.org/docs/2.15/generated/torch.optim.Adam.html).
- Gardez le même taux d'apprentissage (`lr=0.1`)
- Observez la vitesse de convergence. Est-elle plus lente ou plus rapide que le SGD sur les premières étapes ?

**Note :** Adam utilise des "moments" (une sorte de mémoire gradients précédents) pour ajuster sa trajectoire. Discuter de la précision de convergence, ajouter des itérations supplémentaires si nécessaire et observer le résultat. Changer la valeur de `lr` et observer le résuat.

In [57]:

from torch.optim import Adam

def adam_with_lr(lr, iterations=50):
    print(f"\nAdam with lr={lr} iterations={iterations}", end="\n\n")
    x = torch.tensor(10.0, requires_grad=True)
    optimizer = Adam([x], lr=lr)
    
    for i in range(iterations):
        optimizer.zero_grad()
        f = x**2 - 4*x + 4          # forward pass: compute f(x)
        f.backward()                # backward pass
    
        optimizer.step()
    
        if (i + 1) % 10 == 0:
            print(f"Iteration {i+1} | x = {x.item():.6f} | f(x) = {f.item():.6f}")

adam_with_lr(0.8111, 100)

adam_with_lr(0.1)
adam_with_lr(0.3)
adam_with_lr(0.8)
adam_with_lr(1)


Adam with lr=0.8111 iterations=100

Iteration 10 | x = 2.523553 | f(x) = 1.312797
Iteration 20 | x = -0.180403 | f(x) = 4.879683
Iteration 30 | x = 1.798940 | f(x) = 0.187448
Iteration 40 | x = 2.808967 | f(x) = 0.692296
Iteration 50 | x = 1.919817 | f(x) = 0.000080
Iteration 60 | x = 1.735371 | f(x) = 0.089715
Iteration 70 | x = 2.132364 | f(x) = 0.012526
Iteration 80 | x = 2.029037 | f(x) = 0.002704
Iteration 90 | x = 1.938035 | f(x) = 0.004524
Iteration 100 | x = 2.028454 | f(x) = 0.000547

Adam with lr=0.1 iterations=50

Iteration 10 | x = 9.004462 | f(x) = 50.456532
Iteration 20 | x = 8.034358 | f(x) = 37.571228
Iteration 30 | x = 7.112843 | f(x) = 27.063196
Iteration 40 | x = 6.258662 | f(x) = 18.841099
Iteration 50 | x = 5.484863 | f(x) = 12.661987

Adam with lr=0.3 iterations=50

Iteration 10 | x = 7.049632 | f(x) = 28.473181
Iteration 20 | x = 4.425385 | f(x) = 7.086887
Iteration 30 | x = 2.509810 | f(x) = 0.435745
Iteration 40 | x = 1.543797 | f(x) = 0.161410
Iteration 50 | 